# Phase 14 - Define the experimental dataset

| | |
| --- | --- |
| **Workflow** | `flood prediction workflow.md` **Phase 14 - Define the experimental dataset**: *create the final modelling table (Location / Timestamp / Predictor 1..N / Target / Prediction horizon). Keep a copy of the raw dataset separately. Never overwrite the original.* |
| **Checklist #** | 26 (experimental / model-ready table) |
| **Input** | `data/processed/flood_dataset_model_ready.csv` (200,665 x 73) + `feature_whitelist.csv` (Phase 13) + `feature_sets.json` (Phase 12) |
| **Boundary** | `reports/phase0_research_definition.md` §5 - issue instant = 24:00 UTC of day `t` |
| **Outputs** | `data/processed/experimental_dataset.csv` (200,665 x 67) · `data/processed/experimental_dataset_dictionary.csv` |
| **Source rule** | `flood_dataset_model_ready.csv` is read-only: SHA-256 before == after (E9) |

Ten checks, **E1-E10**. This table is what Phases 15-16 split and train on; it is written
by *selecting columns*, never by recomputing them, so every value is byte-identical to the
audited source.

In [1]:
import hashlib, json, time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(r"D:\Flood predictor")
PROC = ROOT / "data" / "processed"
SRC = PROC / "flood_dataset_model_ready.csv"
HORIZONS = ["label_24h", "label_48h", "label_72h"]

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for blk in iter(lambda: f.read(1 << 20), b""):
            h.update(blk)
    return h.hexdigest()

SRC_SHA = sha256(SRC)
mr = pd.read_csv(SRC)
mr["date"] = pd.to_datetime(mr["date"])
WL = pd.read_csv(PROC / "feature_whitelist.csv")
sets = json.loads((PROC / "feature_sets.json").read_text(encoding="utf-8"))
FULL = list(sets["sets"]["FULL"])
BANNED = list(sets["banned"])

RES = []
def check(cid, detail, ok=True):
    RES.append({"id": cid, "pass": bool(ok)})
    print(f"[{'PASS' if ok else 'FAIL'}] {cid}: {detail}")
    assert ok, cid

print(f"source {mr.shape} | whitelist {WL.shape} | Phase-12 FULL {len(FULL)} | "
      f"source sha256 {SRC_SHA[:16]}...")

source (200665, 73) | whitelist (73, 7) | Phase-12 FULL 62 | source sha256 15a0b78518cba0ca...


## What the workflow asks for

> *At this point create the **final modelling table**. Conceptually: Location, Timestamp,
> Predictor 1 ... Predictor N, Target, Prediction horizon. Keep a copy of the raw dataset
> separately. Never overwrite the original.* (workflow **Phase 14**)

So the table has exactly four column blocks:

| block | columns | count |
| --- | --- | --- |
| **Location** | `station_id` (join key for `lat`/`lon` and for spatial validation in Phase 15 step 26) | 1 |
| **Timestamp** | `date` (the day `t` the prediction is issued for) | 1 |
| **Predictors** | the Phase-12 `FULL` set = the Phase-13 whitelist's `ADMIT` + `ADMIT-WATCH` columns | 62 |
| **Target** | `label_24h` `label_48h` `label_72h` (the three prediction horizons, one column each) | 3 |

Everything else in the 73-column source is **deliberately excluded**: the 10 banned
columns minus the 3 targets (`name`, `source`, `obs_days_24h/48h/72h` are identity or
label-window information), plus the 1 constant that Phase 12 dropped (`day_observed`).
No value is recalculated - only columns are selected, so the audit of Phase 13 still holds
for every cell of this table.

In [2]:
# E1 - provenance: the three artifacts that define the table agree with each other
assert mr.shape == (200665, 73), mr.shape
assert WL.shape[0] == 73 and set(WL["column"]) == set(mr.columns)
counts = WL["ruling"].value_counts().to_dict()
assert counts == {"ADMIT": 61, "BAN": 10, "ADMIT-WATCH": 1, "ADMIT-CONSTANT": 1}, counts
admit = set(WL.loc[WL["ruling"].isin(["ADMIT", "ADMIT-WATCH"]), "column"])
assert admit == set(FULL), admit ^ set(FULL)
assert set(BANNED) == {"station_id", "name", "date", "source",
                       "label_24h", "label_48h", "label_72h",
                       "obs_days_24h", "obs_days_48h", "obs_days_72h"}
assert "phase13" in sets and "phase12" not in sets  # whitelist ruling recorded downstream
assert set(FULL) <= set(mr.columns) and set(HORIZONS) <= set(mr.columns)

check("E1 provenance",
      f"source {mr.shape} + whitelist 73 rows {counts} + Phase-12 FULL {len(FULL)} are "
      f"mutually consistent; the 10 banned columns and the 3 targets are all present, "
      f"so every exclusion below is a choice")

[PASS] E1 provenance: source (200665, 73) + whitelist 73 rows {'ADMIT': 61, 'BAN': 10, 'ADMIT-WATCH': 1, 'ADMIT-CONSTANT': 1} + Phase-12 FULL 62 are mutually consistent; the 10 banned columns and the 3 targets are all present, so every exclusion below is a choice


In [3]:
# E2 - schema contract: Location | Timestamp | 62 predictors | 3 targets
KEYS = ["station_id", "date"]
PRED = list(FULL)
TARGETS = list(HORIZONS)
ORDER = KEYS + PRED + TARGETS

assert ORDER[:2] == KEYS and ORDER[-3:] == TARGETS
assert len(ORDER) == 67 and len(set(ORDER)) == 67
assert set(PRED) == admit                      # ADMIT + ADMIT-WATCH, nothing else
assert not (set(PRED) & set(BANNED))           # banned columns never re-enter as features
assert set(TARGETS) == set(BANNED) & set(HORIZONS)
assert "day_observed" not in PRED              # ADMIT-CONSTANT, dropped at Phase 12 step 18a

check("E2 schema contract",
      f"67 columns = 1 location + 1 timestamp + {len(PRED)} predictors (FULL = whitelist "
      f"ADMIT 61 + ADMIT-WATCH 1) + {len(TARGETS)} targets; 0 banned columns among the "
      f"predictors")

[PASS] E2 schema contract: 67 columns = 1 location + 1 timestamp + 62 predictors (FULL = whitelist ADMIT 61 + ADMIT-WATCH 1) + 3 targets; 0 banned columns among the predictors


In [4]:
# E3 - exactly 6 columns leave the source, each for a stated reason
REMOVED = {
    "name": "identity/date key (banned); redundant with station_id",
    "source": "identity/date key (banned)",
    "obs_days_24h": "label-window information (banned) - label availability is visible "
                    "directly from the target being NaN",
    "obs_days_48h": "label-window information (banned)",
    "obs_days_72h": "label-window information (banned)",
    "day_observed": "ADMIT-CONSTANT (constant in the model-ready table, dropped at "
                    "Phase 12 step 18a)",
}
assert set(mr.columns) - set(ORDER) == set(REMOVED), set(mr.columns) ^ set(ORDER)
assert len(REMOVED) == 6 and 73 - 6 == 67

# the table is a pure column selection: every retained cell is byte-identical to the source
EXPL = mr[ORDER].copy()
assert EXPL.shape == (200665, 67)
assert all(EXPL[c].equals(mr[c]) for c in ORDER)

check("E3 column selection",
      "6 columns removed (" + ", ".join(REMOVED) + "); all 67 retained columns are "
      "identical to the source cells - no recomputation, so the Phase-13 audit carries over")

[PASS] E3 column selection: 6 columns removed (name, source, obs_days_24h, obs_days_48h, obs_days_72h, day_observed); all 67 retained columns are identical to the source cells - no recomputation, so the Phase-13 audit carries over


In [5]:
# E4 - row and key contract
assert len(EXPL) == 200665
assert EXPL.duplicated(KEYS).sum() == 0
n_st = EXPL["station_id"].nunique()
assert n_st == 133, n_st
dmin, dmax = EXPL["date"].min(), EXPL["date"].max()
assert dmin == pd.Timestamp("2017-01-06") and dmax == pd.Timestamp("2026-10-05")

# strictly increasing calendar time within every station, never a duplicate day
srt = EXPL.sort_values(["station_id", "date"])
assert int((srt.groupby("station_id")["date"].diff().dt.days.dropna() <= 0).sum()) == 0
# the 2018 archive hole never entered the model-ready table (Phase 13 L10)
assert int((EXPL["date"].dt.year == 2018).sum()) == 0

yrs = EXPL.groupby(EXPL["date"].dt.year).size()
check("E4 rows and keys",
      f"{len(EXPL):,} rows, {EXPL.duplicated(KEYS).sum()} duplicate station-days, "
      f"{n_st} stations, {dmin.date()} -> {dmax.date()}, time strictly increasing within "
      f"every station, 0 rows in 2018; years {dict(yrs)}")

[PASS] E4 rows and keys: 200,665 rows, 0 duplicate station-days, 133 stations, 2017-01-06 -> 2026-10-05, time strictly increasing within every station, 0 rows in 2018; years {2017: np.int64(17453), 2019: np.int64(18309), 2020: np.int64(20055), 2021: np.int64(21754), 2022: np.int64(21920), 2023: np.int64(23869), 2024: np.int64(24363), 2025: np.int64(28252), 2026: np.int64(24690)}


In [6]:
# E5 - dtype contract
num_pred = EXPL[PRED].select_dtypes(include=[np.number]).columns.tolist()
text_pred = [c for c in PRED if c not in num_pred]
assert set(text_pred) == {"river", "basin", "division", "district", "upazilla"}, text_pred
assert len(num_pred) + len(text_pred) == 62
assert pd.api.types.is_datetime64_any_dtype(EXPL["date"])
assert np.issubdtype(EXPL["station_id"].dtype, np.integer)
for h in TARGETS:
    vals = set(EXPL[h].dropna().unique())
    assert vals <= {0.0, 1.0}, (h, vals)
    assert EXPL[h].isna().any()

check("E5 dtypes",
      f"predictors: {len(num_pred)} numeric + {len(text_pred)} text "
      f"({', '.join(text_pred)} - frequency-encoded later, raw in this table); keys: int64 + "
      f"datetime64; targets: {len(TARGETS)} binary float columns with NaN for unlabelled "
      f"windows")

[PASS] E5 dtypes: predictors: 57 numeric + 5 text (river, basin, division, district, upazilla - frequency-encoded later, raw in this table); keys: int64 + datetime64; targets: 3 binary float columns with NaN for unlabelled windows


## Missingness of the final table

Two different kinds of NaN, and Phase 15/16 must not confuse them:

* **gap-driven NaN** - a value that *would* exist but the observation record does not
  reach back far enough (the 2018 archive hole behind `wl_last_lag1..7`);
* **structural NaN** - a value that *cannot* exist for a station (no upstream gauge
  `up_*`, no rain weather station, no official danger level).

Every model here is a tree model first (Phase 16+), so NaN is kept as *unknown*, never
filled - except in the LINEAR set, where imputation is a Phase-16 decision.

In [7]:
# E6 - missingness profile of the 67-column table
def pct(c):
    return float(EXPL[c].isna().mean())

prof = pd.DataFrame({
    "column": ORDER,
    "block": (["location", "timestamp"] + ["predictor"] * 62 + ["target"] * 3),
    "dtype": [str(EXPL[c].dtype) for c in ORDER],
    "pct_null": [pct(c) for c in ORDER],
})
prof["pct_null"] = (prof["pct_null"] * 100).round(2)

pred_null = prof[prof["block"] == "predictor"].sort_values("pct_null", ascending=False)
zero_nan = pred_null[pred_null["pct_null"] == 0.0]["column"].tolist()
full_nan = [c for c in PRED if EXPL[c].isna().all()]
assert not full_nan, full_nan
up = [c for c in PRED if c.startswith("up_")]
assert all(0.60 <= pct(c) <= 0.80 for c in up), {c: round(pct(c), 3) for c in up}
assert pct("wl_last_lag1") > 0          # the archive hole leaves a mark

print(f"predictors with 0% NaN: {len(zero_nan)}/62 -> {zero_nan}")
print("\npredictors by missingness (top 20):")
print(pred_null.head(20).to_string(index=False))
print("\ntargets:")
print(prof[prof['block'] == 'target'].to_string(index=False))

check("E6 missingness profile",
      f"0 fully-empty predictors; {len(zero_nan)} predictors are complete; structural NaN "
      f"up_* {min(pct(c) for c in up):.0%}-{max(pct(c) for c in up):.0%}; wl_last_lag1 "
      f"{pct('wl_last_lag1'):.1%} (2018 hole); targets NaN as contracted")

predictors with 0% NaN: 37/62 -> ['wl_last', 'basin', 'division', 'district', 'river', 'dist_river_km', 'elevation_m', 'dist_to_coast_km', 'slope_deg', 'river_density_10km_km_per_km2', 'dist_major_river_km', 'n_river_lines_10km', 'lat', 'lon', 'danger_level', 'n_readings', 'wl_mean', 'wl_max', 'wl_min', 'wl_first', 'wl_dist_to_danger', 'wet_days_7', 'rain_max_7d', 'rain_30d', 'rain_14d', 'rain_7d', 'rain_3d', 'wl_pct_danger', 'wl_exceed_days_30', 'nb5_rain_7d', 'nb3_rain_14d', 'nb3_rain_7d', 'doy_cos', 'doy_sin', 'nb5_n_wl_danger', 'nb5_rain_14d', 'nb3_n_wl_danger']

predictors by missingness (top 20):
               column     block   dtype  pct_null
up_wl_pct_danger_lag2 predictor float64     70.45
up_wl_pct_danger_lag1 predictor float64     70.37
       up_rain7d_lag2 predictor float64     67.14
       up_rain7d_lag1 predictor float64     67.13
        wl_max_roll30 predictor float64      3.53
         wl_last_lag7 predictor float64      3.37
         wl_last_lag6 predictor float64 

In [8]:
# E7 - target contract, re-checked against phase0 section 4
EXPECT = {"label_24h": (200559, 9982, 106, 4.98),
          "label_48h": (197748, 11241, 2917, 5.68),
          "label_72h": (195174, 12340, 5491, 6.32)}
rows = []
for h in TARGETS:
    n = int(EXPL[h].notna().sum())
    pos = int(EXPL[h].sum())
    nan = int(EXPL[h].isna().sum())
    rate = round(100 * pos / n, 2)
    exp_n, exp_pos, exp_nan, exp_rate = EXPECT[h]
    assert (n, pos, nan) == (exp_n, exp_pos, exp_nan), (h, n, pos, nan)
    assert abs(rate - exp_rate) < 0.01, (h, rate, exp_rate)
    rows.append({"target": h, "n": n, "positives": pos, "base_rate_%": rate, "NaN": nan,
                 "imbalance_0:1": round((n - pos) / pos, 1)})
TGT = pd.DataFrame(rows)
assert TGT["n"].is_monotonic_decreasing and TGT["positives"].is_monotonic_increasing

print(TGT.to_string(index=False))
check("E7 target contract",
      "label_24h/48h/72h = 200,559 / 197,748 / 195,174 labelled rows with 9,982 / 11,241 / "
      "12,340 positives (4.98 / 5.68 / 6.32 %) - matches phase0 §4 exactly; PR-AUC is the "
      "headline metric (rare, not degenerate)")

   target      n  positives  base_rate_%  NaN  imbalance_0:1
label_24h 200559       9982         4.98  106           19.1
label_48h 197748      11241         5.68 2917           16.6
label_72h 195174      12340         6.32 5491           14.8
[PASS] E7 target contract: label_24h/48h/72h = 200,559 / 197,748 / 195,174 labelled rows with 9,982 / 11,241 / 12,340 positives (4.98 / 5.68 / 6.32 %) - matches phase0 §4 exactly; PR-AUC is the headline metric (rare, not degenerate)


## Writing the table

The source (`flood_dataset_model_ready.csv`) is **never** written: the notebook records its
SHA-256 first (setup cell), writes `experimental_dataset.csv`, then re-reads it and proves

1. round-trip equality - every value survives the CSV, order of columns preserved;
2. source immutability - the SHA-256 of the input is byte-identical after the run.

In [9]:
# E8 - write, re-read, compare
OUTCSV = PROC / "experimental_dataset.csv"
t0 = time.time()
EXPL.to_csv(OUTCSV, index=False)
back = pd.read_csv(OUTCSV)
back["date"] = pd.to_datetime(back["date"])
assert list(back.columns) == ORDER, "column order lost"
pd.testing.assert_frame_equal(EXPL.reset_index(drop=True), back, check_dtype=False)
size = OUTCSV.stat().st_size

check("E8 write + round-trip",
      f"{OUTCSV.name}: {back.shape[0]:,} x {back.shape[1]} written and re-read in "
      f"{time.time() - t0:.1f}s ({size / 1e6:.1f} MB); values, dtypes (after inference) and "
      f"column order identical - no silent coercion")

[PASS] E8 write + round-trip: experimental_dataset.csv: 200,665 x 67 written and re-read in 44.6s (125.7 MB); values, dtypes (after inference) and column order identical - no silent coercion


In [10]:
# E9 - immutability of the source (workflow: never overwrite the original)
assert sha256(SRC) == SRC_SHA, "source file changed"
recheck = pd.read_csv(SRC, usecols=["station_id"]).shape[0]
assert recheck == 200665
check("E9 source immutability",
      f"SHA-256 of flood_dataset_model_ready.csv unchanged ({SRC_SHA[:16]}..., "
      f"{SRC.stat().st_size / 1e6:.1f} MB); the raw copy stays as it was")

[PASS] E9 source immutability: SHA-256 of flood_dataset_model_ready.csv unchanged (15a0b78518cba0ca..., 138.0 MB); the raw copy stays as it was


In [11]:
# E10 - data dictionary + preview
GRP = dict(zip(WL["column"], WL["group"]))
LATEST = dict(zip(WL["column"], WL["latest_day_used"]))
RULE = dict(zip(WL["column"], WL["ruling"]))
BLOCK = {c: "location" if c == "station_id" else "timestamp" if c == "date"
         else "target" for c in ORDER}
BLOCK.update({c: "predictor" for c in PRED})
ROLE = {c: "key - never a feature" for c in KEYS}
ROLE.update({c: "feature (Phase-12 FULL)" for c in PRED})
ROLE.update({c: "target" for c in TARGETS})

DICT = pd.DataFrame({
    "position": range(1, 68),
    "block": [BLOCK[c] for c in ORDER],
    "column": ORDER,
    "role": [ROLE[c] for c in ORDER],
    "whitelist_group": [GRP.get(c, "target" if c in TARGETS else "") for c in ORDER],
    "latest_day_used": [LATEST.get(c, "t+1 .. t+h") for c in ORDER],
    "ruling": [RULE.get(c, "TARGET") for c in ORDER],
    "dtype": [str(EXPL[c].dtype) for c in ORDER],
    "non_null": [int(EXPL[c].notna().sum()) for c in ORDER],
    "pct_null": [round(pct(c) * 100, 2) for c in ORDER],
})
DICT.to_csv(PROC / "experimental_dataset_dictionary.csv", index=False)
assert len(DICT) == 67 and (DICT["block"] == "predictor").sum() == 62

pd.set_option("display.max_rows", 70, "display.width", 200)
print(DICT.to_string(index=False))
print(f"\nexperimental_dataset.csv + experimental_dataset_dictionary.csv written "
      f"({size / 1e6:.1f} MB, {EXPL.memory_usage(deep=True).sum() / 1e6:.1f} MB in memory)")
check("E10 data dictionary",
      "67-row dictionary written (position, block, role, whitelist group, latest day used, "
      "ruling, dtype, null profile)")

 position     block                        column                    role                                      whitelist_group           latest_day_used      ruling          dtype  non_null  pct_null
        1  location                    station_id   key - never a feature                                  identity / date key                       key         BAN          int64    200665      0.00
        2 timestamp                          date   key - never a feature                                  identity / date key                       key         BAN datetime64[us]    200665      0.00
        3 predictor                         river feature (Phase-12 FULL)                       identity / published threshold static (station-constant)       ADMIT            str    200665      0.00
        4 predictor                         basin feature (Phase-12 FULL)                       identity / published threshold static (station-constant)       ADMIT            str    200665      0.00



experimental_dataset.csv + experimental_dataset_dictionary.csv written (125.7 MB, 157.1 MB in memory)
[PASS] E10 data dictionary: 67-row dictionary written (position, block, role, whitelist group, latest day used, ruling, dtype, null profile)


In [12]:
print("=" * 78)
print("FINAL EXPERIMENTAL DATASET  data/processed/experimental_dataset.csv")
print("=" * 78)
print(f"shape            : {EXPL.shape[0]:,} rows x {EXPL.shape[1]} columns")
print(f"location keys    : 133 stations (station_id; lat/lon are predictors)")
print(f"timestamps       : {dmin.date()} .. {dmax.date()} (day t, one row per observed day)")
print(f"predictors       : 62 (Phase-12 FULL = Phase-13 ADMIT 61 + ADMIT-WATCH 1)")
print(f"targets          : label_24h / label_48h / label_72h")
print(f"dropped from src : {', '.join(sorted(REMOVED))}")
print(f"split            : NOT assigned here - Phase 15 decides train/val/test")
print("\nfirst 3 rows (transposed):")
print(EXPL.head(3).T.to_string())
print("\nall checks:")
for r in RES:
    print(f"  {'PASS' if r['pass'] else 'FAIL'}  {r['id']}")
print("ALL PHASE-14 CHECKS PASSED")

FINAL EXPERIMENTAL DATASET  data/processed/experimental_dataset.csv
shape            : 200,665 rows x 67 columns
location keys    : 133 stations (station_id; lat/lon are predictors)
timestamps       : 2017-01-06 .. 2026-10-05 (day t, one row per observed day)
predictors       : 62 (Phase-12 FULL = Phase-13 ADMIT 61 + ADMIT-WATCH 1)
targets          : label_24h / label_48h / label_72h
dropped from src : day_observed, name, obs_days_24h, obs_days_48h, obs_days_72h, source
split            : NOT assigned here - Phase 15 decides train/val/test

first 3 rows (transposed):
                                                 0                    1                    2
station_id                                       1                    1                    1
date                           2017-05-22 00:00:00  2017-05-23 00:00:00  2017-05-24 00:00:00
river                                   Matamuhuri           Matamuhuri           Matamuhuri
basin                                      SE Hill    